In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# Cell 1: Install dependencies
!pip install -q tiktoken datasets tqdm

In [4]:
# Cell 2: Data Preparation
import os
import numpy as np
import tiktoken
from datasets import load_dataset
from tqdm import tqdm

# Load a small subset of TinyStories
dataset = load_dataset("roneneldan/TinyStories", split='train[:5000]') # 5000 stories for speed
split_dataset = dataset.train_test_split(test_size=0.1)

enc = tiktoken.get_encoding("gpt2")

def process(example):
    ids = enc.encode_ordinary(example['text']) 
    ids.append(enc.eot_token) # append End of Text token
    return {'ids': ids, 'len': len(ids)}

# Tokenize
tokenized = split_dataset.map(process, remove_columns=['text'], num_proc=4)

# Save to binary files
for split, dset in tokenized.items():
    arr_len = np.sum(dset['len'])
    filename = f'{split}.bin'
    arr = np.memmap(filename, dtype=np.uint16, mode='w+', shape=(arr_len,))
    
    idx = 0
    for example in tqdm(dset['ids'], desc=f'writing {filename}'):
        arr[idx : idx + len(example)] = example
        idx += len(example)
    arr.flush()

print("Data preparation complete! train.bin and test.bin created.")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/4500 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/500 [00:00<?, ? examples/s]

writing test.bin: 100%|██████████| 500/500 [00:00<00:00, 6566.82it/s]

Data preparation complete! train.bin and test.bin created.


In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# --- Hyperparameters ---
batch_size = 64 # Increased due to Flash Attention memory efficiency
block_size = 256 
max_iters = 2000
eval_interval = 200
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.1
vocab_size = 50257 # Matches pre-trained tokenizer

# --- Architectural Components ---

class CausalSelfAttention(nn.Module):
    """ Vectorized Multi-Head Attention with Flash Attention """
    def __init__(self):
        super().__init__()
        assert n_embd % n_head == 0
        # Key, Query, Value projections for all heads in one batch
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        # Output projection
        self.c_proj = nn.Linear(n_embd, n_embd)
        # Regularization
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd

    def forward(self, x):
        B, T, C = x.size() 

        # Calculate q, k, v for all heads: (B, T, n_embd) -> (B, T, 3 * n_embd)
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # Reshape for multi-head: (B, nh, T, hs)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        # Flash Attention (Scaled Dot Product Attention)
        # Replaces manual masking and softmax with fused CUDA kernels
        y = F.scaled_dot_product_attention(
            q, q, v, 
            attn_mask=None, 
            dropout_p=dropout if self.training else 0, 
            is_causal=True
        )

        # Re-assemble heads: (B, nh, T, hs) -> (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C) 

        # Output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

class FeedForward(nn.Module):
    """ A simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=False),
            nn.GELU(), # Upgraded from ReLU
            nn.Linear(4 * n_embd, n_embd, bias=False),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

# --- The NanoGPT Model ---

class NanoGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(vocab_size, n_embd),
            wpe = nn.Embedding(block_size, n_embd),
            drop = nn.Dropout(dropout),
            h = nn.ModuleList([Block() for _ in range(n_layer)]),
            ln_f = nn.LayerNorm(n_embd),
        ))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tieing: https://arxiv.org/abs/1608.05859
        self.transformer.wte.weight = self.lm_head.weight

        # Init weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        B, T = idx.size()
        assert T <= block_size, f"Cannot forward sequence of length {T}, block size is {block_size}"
        
        # Embeddings
        pos = torch.arange(0, T, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx) 
        pos_emb = self.transformer.wpe(pos) 
        
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # Training mode
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # Inference mode: only compute logits for the last position
            logits = self.lm_head(x[:, [-1], :]) 
            loss = None

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            # Crop context if it exceeds block_size
            idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
            # Forward the model
            logits, _ = self(idx_cond)
            # Pluck the logits at the final step and scale by temperature
            logits = logits[:, -1, :] / temperature
            # Optional top-k filtering
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # Apply softmax
            probs = F.softmax(logits, dim=-1)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # Append to sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [6]:
# Cell 4: Multi-GPU Training Engine
import numpy as np

# 1. Load data
train_data = np.memmap('train.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('test.bin', dtype=np.uint16, mode='r')

# 2. Double batch size for 2 GPUs
batch_size = 64 

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+block_size+1]).astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

# 3. Initialize and Wrap Model
model = NanoGPT()

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    # This splits the 64 batch into 32 per GPU
    model = nn.DataParallel(model)

model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# 4. Training Loop
for iter in range(max_iters):
    if iter % eval_interval == 0:
        print(f"step {iter}: computing loss...")
        
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    loss = loss.mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 500 == 0:
        # Note: we use .mean() in case DataParallel returns a vector of losses
        print(f"Iter {iter}, Loss: {loss.mean().item():.4f}")

# 5. Save correctly (handling the DataParallel wrapper)
if isinstance(model, nn.DataParallel):
    torch.save(model.module.state_dict(), 'nanogpt.pt')
else:
    torch.save(model.state_dict(), 'nanogpt.pt')

print("Training finished and model saved!")

Using 2 GPUs!
step 0: computing loss...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Iter 0, Loss: 10.8497
step 200: computing loss...
step 400: computing loss...
Iter 500, Loss: 4.0486
step 600: computing loss...
step 800: computing loss...
step 1000: computing loss...
Iter 1000, Loss: 3.4930
step 1200: computing loss...
step 1400: computing loss...
Iter 1500, Loss: 3.2643
step 1600: computing loss...
step 1800: computing loss...
Training finished and model saved!


In [10]:
#Validation

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            out[split] = losses.mean()
    model.train()
    return out

In [11]:
#Sanity check for the dataset.

xb, yb = get_batch('train')
print(f"Batch shape: {xb.shape}")
print(f"First sequence tokens: {xb[0][:10]}")
print(f"Decoded: {enc.decode(xb[0][:10].tolist())}")

Batch shape: torch.Size([64, 256])
First sequence tokens: tensor([ 3772,    13,  3574,   326,  1110,   319,    11, 20037,  3088,   284],
       device='cuda:0')
Decoded:  happy. From that day on, Lily tried to


In [12]:
# Start with a prompt
context = torch.zeros((1, 1), dtype=torch.long, device=device) # or encode a prompt like "Once upon"
print(enc.decode(model.module.generate(context, max_new_tokens=100)[0].tolist()))

! But then, Out walked over the couch would not talking. So they quickly opened their neighbours about their advice and some ice cream, because one was just as warm he wasn't respectful.

The tire was so excited to go to the cube and climb it never seen again. Inside the tree was a magic cave, but it was so sweet and didn't forget what to make it enjoyed it. 

Joe was very nice. He was so happy, but could work he made the pile
